In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
import json
import torch
import pandas as pd
import gradio as gr

In [ ]:
AVAILABLE_MODELS = {
    "Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct",
    "Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct",
    "Llama-3.1-8B-Instruct": "meta-llama/Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3": "mistralai/Mistral-7B-Instruct-v0.3",
    "Gemma-2-9B-It": "google/gemma-2-9b-it"
}

In [ ]:
model = None
tokenizer = None
current_model = None

In [ ]:
def load_model(model_name):
    """
    Load the selected model if it is not already loaded.
    """

    global model, tokenizer, current_model

    if current_model == model_name:
        return model, tokenizer

    model_path = AVAILABLE_MODELS[model_name]

    print(f"Loading model: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )

    current_model = model_name
    return model, tokenizer

In [ ]:
def generate_result(
    model_name,
    instruction,
    samples,
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
):
    """
    Generate a synthetic dataset using the selected model.
    """

    model, tokenizer = load_model(model_name)

    prompt = f"""
    You are an expert synthetic dataset generator.

    Task:
    {instruction}

    Generate exactly {samples} examples.

    Return ONLY valid JSON.

    Example:

    [
        {{
            "input": "...",
            "output": "..."
        }}
    ]
    """

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        chat,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_tokens,
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1] :],
        skip_special_tokens=True,
    )

    return response

In [ ]:
def generate_data(
    model_name,
    instruction,
    samples,
    temperature,
    top_p,
    max_tokens,
):

    result = generate_result(
        model_name=model_name,
        instruction=instruction,
        samples=samples,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )

    try:
        parsed = json.loads(result)

        df = pd.DataFrame(parsed)

        csv_path = "synthetic_data.csv"

        df.to_csv(csv_path, index=False)

        json_path = "synthetic_data.json"

        with open(json_path, "w") as f:
            json.dump(parsed, f, indent=4)

        return (
            result,
            df,
            json_path,
            csv_path
        )

    except Exception:

        return (
            result,
            None,
            None,
            None
        )

In [ ]:
with gr.Blocks(title="Synthetic Dataset Generator") as demo:

    gr.Markdown(
        "# Synthetic Dataset Generator using HuggingFace LLMs"
    )

    with gr.Row():

        model = gr.Dropdown(
            choices=list(AVAILABLE_MODELS.keys()),
            value="Qwen2.5-3B-Instruct",
            label="Model"
        )

        samples = gr.Slider(
            1,
            100,
            value=10,
            step=1,
            label="Samples"
        )

    instruction = gr.Textbox(
        lines=8,
        label="Dataset Description",
        placeholder="""
            Generate customer support conversations
            Generate medical QA
            Generate finance instruction dataset
            Generate email classification data
            """
    )

    with gr.Row():

        temperature = gr.Slider(
            0,
            2,
            value=0.8,
            label="Temperature"
        )

        top_p = gr.Slider(
            0,
            1,
            value=0.95,
            label="Top P"
        )

        max_tokens = gr.Slider(
            128,
            2048,
            value=1024,
            step=64,
            label="Max Tokens"
        )

    generate = gr.Button(
        "Generate Dataset",
        variant="primary"
    )

    output = gr.Textbox(
        label="Generated JSON",
        lines=20
    )

    dataframe = gr.Dataframe()

    json_file = gr.File(label="Download JSON")

    csv_file = gr.File(label="Download CSV")

    generate.click(
        generate_data,
        inputs=[
            model,
            instruction,
            samples,
            temperature,
            top_p,
            max_tokens
        ],
        outputs=[
            output,
            dataframe,
            json_file,
            csv_file
        ]
    )

demo.launch()